### `Imports`

In [227]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

### `Load train and Kaggle test data`

In [228]:
train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (8693, 14)
Test: (4277, 13)


In [229]:
train_df.columns

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name', 'Transported'],
      dtype='str')

In [230]:
# separate features and target variable from training data
X_known = train_df.drop(columns=["Transported"])
y_known = train_df["Transported"]

X_test = test_df.copy()

### `Feature engineering`

In [231]:
X_test.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


In [232]:
X_test.PassengerId

0       0013_01
1       0018_01
2       0019_01
3       0021_01
4       0023_01
         ...   
4272    9266_02
4273    9269_01
4274    9271_01
4275    9273_01
4276    9277_01
Name: PassengerId, Length: 4277, dtype: str

In [233]:
X_known.PassengerId

0       0001_01
1       0002_01
2       0003_01
3       0003_02
4       0004_01
         ...   
8688    9276_01
8689    9278_01
8690    9279_01
8691    9280_01
8692    9280_02
Name: PassengerId, Length: 8693, dtype: str

In [234]:
X_known.PassengerId

0       0001_01
1       0002_01
2       0003_01
3       0003_02
4       0004_01
         ...   
8688    9276_01
8689    9278_01
8690    9279_01
8691    9280_01
8692    9280_02
Name: PassengerId, Length: 8693, dtype: str

In [235]:
X_known.columns

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name'],
      dtype='str')

In [236]:
def feature_engineer(df):
    df = df.copy() # We do not want to modify the actual dataFrame

    # Extract group ID
    df["GroupID"] = df["PassengerId"].str.split("_").str[0]

    # Extract group size
    group_count = df["GroupID"].value_counts()
    df["GroupSize"] = df["GroupID"].map(group_count)

    spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

    # Create a total spending column
    df["TotalSpending"] = df[spending_cols].sum(axis=1)

    # Extract Cabin into three new useful columns
    df[['Deck', 'CabinNumber', 'Side']] = df['Cabin'].str.split('/', expand=True)

    # Convert CabinNumber to numeric, coercing errors to NaN
    df["CabinNumber"] = pd.to_numeric(
                            df["CabinNumber"],
                            errors="coerce")


    # Drop unecessaary columns 
    df = df.drop(columns=["GroupID","PassengerId","Cabin","Name"])
    

    return df

### `Define preprocessor that includes pipelines`

In [237]:
def build_preprocessor(dataFrame):

    numerical_features = dataFrame.select_dtypes(include="number").columns.tolist()
    categorical_features = dataFrame.select_dtypes(exclude="number").columns.tolist()

    numerical_pipeline = Pipeline(
        steps=[
            ("imputer",SimpleImputer(strategy="median")),
            ("scalar", StandardScaler())
        ]
    )

    # Categorical pipeline

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer",SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]
    )

    # Combine both pipeline

    preprocessor_ = ColumnTransformer(
        transformers=[
            ("numerical", 
            numerical_pipeline,
            numerical_features),

            ("categorical",
            categorical_pipeline,
            categorical_features)
        ]
    )

    return preprocessor_ 

In [238]:
X_known_engineered = feature_engineer(X_known)

X_test_engineered = feature_engineer(X_test)

preprocessor = build_preprocessor(X_known_engineered)

X_known_preprocessed = preprocessor.fit_transform(X_known_engineered)
X_test_preprocessed = preprocessor.transform(X_test_engineered)

In [239]:
print("Full training data:", X_known_preprocessed.shape)
print("Kaggle test data:", X_test_preprocessed.shape)

# Check for NaN values in the preprocessed data
print("Training NaNs:", np.isnan(X_known_preprocessed).sum())
print("Test NaNs:", np.isnan(X_test_preprocessed).sum())

# Check the type of the preprocessed data
print("Training data type:", type(X_known_preprocessed))

Full training data: (8693, 29)
Kaggle test data: (4277, 29)
Training NaNs: 0
Test NaNs: 0
Training data type: <class 'numpy.ndarray'>


### `Rebuild chosen neural network`

In [244]:
final_model = Sequential([
    Dense(16, activation='relu', input_shape=(X_known_preprocessed.shape[1],)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

In [241]:
# Compile the model with appropriate loss function and optimizer for binary classification
final_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [242]:
# Train the final model
history = final_model.fit(
    X_known_preprocessed,
    y_known,
    epochs=30,
    batch_size=32
)

Epoch 1/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 1s 714us/step - accuracy: 0.7283 - loss: 0.5383
Epoch 2/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 0s 738us/step - accuracy: 0.7872 - loss: 0.4365
Epoch 3/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 0s 813us/step - accuracy: 0.7937 - loss: 0.4246
Epoch 4/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 0s 718us/step - accuracy: 0.7986 - loss: 0.4183
Epoch 5/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 0s 720us/step - accuracy: 0.7988 - loss: 0.4158
Epoch 6/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 0s 721us/step - accuracy: 0.8028 - loss: 0.4113
Epoch 7/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 0s 787us/step - accuracy: 0.8031 - loss: 0.4089
Epoch 8/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 0s 739us/step - accuracy: 0.8027 - loss: 0.4062
Epoch 9/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 0s 727us/step - accuracy: 0.8036 - loss: 0.4045
Epoch 10/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 0s 754us/step - accuracy: 0.8059 - loss: 0.4025
Epoch 11/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 0s 715us/step - accuracy: 0.8057 - loss: 0.4006
Epoch 12/30
272/272 ━━━━━━━━━━

### `Save the actual final model and preprocessor`

In [243]:
final_model.save("../models/final_spaceship_titanic.keras")

print("Final model saved successfully.")

import joblib

joblib.dump(
    preprocessor,
    "../models/preprocessor.joblib"
)

print("Preprocessor saved successfully.")

Final model saved successfully.
Preprocessor saved successfully.
